# Extended GPU KV-cache evaluation
Use a T4 runtime. This notebook runs **150 requests per strategy, 3 repeats, 3 workloads, and 2 cache budgets**: 90 runs / 13,500 request executions. Fixed output length is four tokens. All runs use float32 and strict output equality; FP16 remains unvalidated.

This is substantially longer than the short check. Keep time available and save the checkpoint ZIP from Colab's Files panel if you must leave. Free runtime availability is not guaranteed. The archive is refreshed after each workload/budget group, including on ordinary failures; abrupt runtime loss can still lose files. The default uses one held-out workload seed (99); repetitions measure timing variability, not independent data samples. Do not tune the predictor on these results.


In [ ]:
import os, pathlib, subprocess, sys
os.environ["USE_TF"] = "0"
repo = pathlib.Path("/content/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git", str(repo)], check=True)
os.chdir(repo)
# Keep Colab's Gradio/Diffusers dependencies intact; reuse its installed CUDA torch.
# virtualenv supplies pip itself; it does not depend on system ensurepip.
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages",
                "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[inference]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select an NVIDIA GPU runtime'; print(torch.cuda.get_device_name())"], check=True)

subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Run the benchmark matrix
Hot: repeated popular prefixes. Shift: popular prefixes change halfway through. Scan: sequential access, a negative control. Retained KV budgets are 8 and 32 MiB; model weights and active tensors need additional GPU memory. Every strategy sees the same trace in each comparison.

Each run gets a new results directory so earlier failed or short-run reports cannot be confused with this experiment. On any correctness failure, the suite stops and archives diagnostics. Download those results rather than bypassing the failure.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
run_dir = Path("/content") / ("kv-cache-suite-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
archive = str(run_dir) + ".zip"
print("Results:", run_dir, flush=True)
subprocess.run([python, "-u", "-m", "kv_cache.suite", "--output", str(run_dir),
                "--requests", "150", "--repeats", "3", "--budgets", "8", "32",
                "--workloads", "hot", "shift", "scan", "--device", "cuda"], check=True)


## Summarize completed runs
The table reports the median of three **per-run p95 values**, not a pooled p95. Check hit rates, correctness and all repetitions before interpreting timing differences. This experiment may show no learned-policy benefit; report that honestly.


In [ ]:
import json, statistics
manifest = json.loads((run_dir / "manifest.json").read_text())
print("Suite status:", manifest["status"])
for name in manifest["completed"]:
    report = json.loads((run_dir / (name + ".json")).read_text())
    assert report["complete"] and report["greedy_outputs_match"]
    print("\n", name)
    for policy in ("none", "fifo", "lru", "lfu", "learned"):
        rows = [r for r in report["runs"] if r["policy"] == policy]
        p95s = [r["ttft_p95_ms"] for r in rows]
        print(f"{policy:8s} hits={rows[0]['hits']}/{rows[0]['requests']} "
              f"median run-p95={statistics.median(p95s):.2f}ms "
              f"range=[{min(p95s):.2f}, {max(p95s):.2f}]ms")


In [ ]:
from google.colab import files
files.download(archive)


If execution stopped at a failed benchmark, run the download cell manually: the suite archives ordinary failures. If the runtime was abruptly terminated, files may be unavailable. Upload the ZIP for analysis. Do not rerun the benchmark cell just to download; that starts a new suite.
